# Ejercicio 2 · ¿Quién sobrevivió al Titanic?


**Módulo 5072 · SAA · UD3 · Métodos ensemble** · 

Tenemos 1.309 pasajeros del Titanic: clase del billete, sexo, edad, familia a bordo, precio del billete y puerto de embarque. **Sobrevivió el 38 %.**

- **Qué predecimos:** `survived` (1 = sobrevive). Es **clasificación**.
- **Métrica:** el **AUC** (0,5 = al azar, 1 = perfecto).

**Qué vas a hacer:** entrenar cinco modelos, compararlos en una tabla y elegir uno.

**Cómo trabajar:**
- Las celdas que dicen **«ya está hecha»**: solo ejecútalas y lee el resultado.
- Las celdas de **«Tu turno»**: cambia cada `___` por lo que falta. Si te atascas, abre la pista.

## 1 · Preparación

Estas celdas ya están hechas: ejecútalas una a una y mira qué sale.

In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

DATA = "../data/"
SEMILLA = 42

# 1. Cargar los datos
df = pd.read_csv(DATA + "titanic.csv")
print(df.shape, "· sobreviven:", round(100 * df["survived"].mean(), 1), "%")

(1309, 14) · sobreviven: 38.2 %


In [22]:
# 2. X (las pistas) e y (si sobrevive). Solo usamos lo que se sabía ANTES del viaje
X = df[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]
y = df["survived"]

In [23]:
# 3. Apartamos el 20 % para el test, con el mismo % de supervivientes en train y en test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA) 

In [24]:
# 4. Huecos en edad y precio: la mediana del TRAIN. Sexo y puerto pasan a números
numeros = ["age", "fare"]
mediana = X_train[numeros].median()
X_train[numeros] = X_train[numeros].fillna(mediana)
X_test[numeros] = X_test[numeros].fillna(mediana)

texto = ["sex", "embarked"]

# Texto a números con OneHotEncoder: aprende las categorías SOLO con el train
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X_train[texto])

def codificar(X):
    cod = pd.DataFrame(ohe.transform(X[texto]), columns=ohe.get_feature_names_out(), index=X.index)
    return pd.concat([X.drop(columns=texto), cod], axis=1)

X_train = codificar(X_train)
X_test = codificar(X_test)

In [25]:
# 5. Validación cruzada: el train en 5 trozos
cv = StratifiedKFold()

In [26]:
# 6. evaluar(): validación cruzada. Entrena el modelo 5 veces, cada vez con 4 trozos,
#    y mide el AUC en el trozo que queda fuera (validación). El test no se toca.
def evaluar(nombre, modelo):
    r = cross_validate(modelo, X_train, y_train, cv=cv, scoring="roc_auc", return_train_score=True)
    return {"modelo": nombre,
            "AUC val": r["test_score"].mean(),
            "± AUC val": r["test_score"].std(),
            "AUC train": r["train_score"].mean()}

def mostrar(resultados):
    return pd.DataFrame(resultados).round(3)

In [27]:
# 7. Baseline: el modelo más tonto. Da a todos la misma probabilidad, así que no ordena nada: AUC = 0,5
resultados = [{"modelo": "Baseline (no ordena)", "AUC val": 0.5, "± AUC val": np.nan, "AUC train": 0.5}]
print("Listo:", len(X_train), "pasajeros para entrenar y validar ·", len(X_test), "para el test")

Listo: 1047 pasajeros para entrenar y validar · 262 para el test


**Qué han hecho estas celdas**
1. Carga los 1.309 pasajeros.
2. Separa `X` (las pistas) e `y` (si sobrevive). Deja fuera:
   - `boat` y `body` (bote salvavidas y número de cuerpo): se saben **después** del hundimiento. Con ellas el modelo haría trampa.
   - `name`, `ticket`, `cabin` y `home.dest`: texto difícil de usar o casi vacío.
3. Aparta el 20 % para el **test**, que no tocamos hasta el final.
4. Rellena los huecos de edad y precio con la **mediana del train**. Pasa `sex` y `embarked` a columnas de 0 y 1 con `OneHotEncoder`, aprendido solo con el train.
5. Prepara la **validación cruzada** de 5 trozos.
6. Crea `evaluar()` y `mostrar()`, como en el gimnasio.
7. Crea la tabla `resultados` con el **baseline**.

**Cómo leer la tabla**
- `AUC val`: el AUC en validación: cómo de bien ordena los casos de más a menos riesgo (0,5 = al azar, 1 = perfecto). Es la columna que usamos para comparar: cuanto **más**, mejor.
- `± AUC val`: cuánto cambia de un trozo de validación a otro. Si dos modelos se diferencian menos que esto, es un **empate**.
- `AUC train`: lo mismo con los datos que el modelo ya ha visto. Si es mucho más alto que `AUC val`, el modelo **memoriza**.

In [28]:
mostrar(resultados)

,modelo,AUC val,± AUC val,AUC train
0,Baseline (no ordena),0.5,NaN,0.5


**Fíjate:** ¿Qué AUC tiene el baseline? ¿Qué quiere decir ese número?

*Tu respuesta:*

Tiene un auc de 0.5 quiere decir que acierta de manera aleatoria (es azar)

## 2 · Tu turno: entrena y compara

**Tarea 1 · Un árbol sin límite.** Cambia `___` por el modelo de árbol: `DecisionTreeClassifier`.

In [29]:
resultados.append(evaluar("Árbol sin límite", DecisionTreeClassifier(random_state=SEMILLA)))
mostrar(resultados)

,modelo,AUC val,± AUC val,AUC train
0,Baseline (no ordena),0.500,NaN,0.500
1,Árbol sin límite,0.735,0.028,0.998


**Fíjate:** Compara `AUC train` y `AUC val`. ¿Memoriza?

*Tu respuesta:*

Si, memoriza. Train 0.998 y val 0.735, hueco 0.26, el árbol se aprende de memoria el train y en val cae

**Tarea 2 · Random Forest de 300 árboles.** Cambia los `___`.

<details><summary><b>Pista</b></summary>

El modelo es `RandomForestClassifier` y el número de árboles va en `n_estimators`.

</details>

In [30]:
resultados.append(evaluar("Random Forest", RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1)))
mostrar(resultados)

,modelo,AUC val,± AUC val,AUC train
0,Baseline (no ordena),0.500,NaN,0.500
1,Árbol sin límite,0.735,0.028,0.998
2,Random Forest,0.823,0.047,0.996


**Fíjate:** ¿Mejora al árbol? ¿La diferencia es mayor que el `±`?

Sí, mejora al árbol. Sube de 0.735 a 0.823 son 0.088 y es más que el ± 0.028 y 0.047 así que sí mejora de verdad.

*Tu respuesta:*

**Tarea 3 · AdaBoost de 200 árboles.** Árboles de una sola pregunta en fila: cada uno se fija más en los casos que fallaron los anteriores. Cambia los `___`.

<details><summary><b>Pista</b></summary>

El modelo es `AdaBoostClassifier` y el número de árboles va en `n_estimators`.

</details>

In [31]:
resultados.append(evaluar("AdaBoost", AdaBoostClassifier(n_estimators=200, random_state=SEMILLA)))
mostrar(resultados)

,modelo,AUC val,± AUC val,AUC train
0,Baseline (no ordena),0.500,NaN,0.500
1,Árbol sin límite,0.735,0.028,0.998
2,Random Forest,0.823,0.047,0.996
3,AdaBoost,0.832,0.022,0.863


**Fíjate:** ¿Mejora al árbol? ¿Memoriza?

*Tu respuesta:*

Si  mejora al árbol de 0.735 a 0.832 son casi 0.10 más que el ±. Y no memoriza casi, train 0.863 y val 0.832 hueco 0.03 muy pequeño

**Tarea 4 · XGBoost por defecto.** Cambia `___` por `XGBClassifier`.

In [32]:
resultados.append(evaluar("XGBoost por defecto", XGBClassifier(random_state=SEMILLA)))
mostrar(resultados)

,modelo,AUC val,± AUC val,AUC train
0,Baseline (no ordena),0.500,NaN,0.500
1,Árbol sin límite,0.735,0.028,0.998
2,Random Forest,0.823,0.047,0.996
3,AdaBoost,0.832,0.022,0.863
4,XGBoost por defecto,0.829,0.042,0.991


**Fíjate:** Compara `AUC train` y `AUC val`. ¿Memoriza?

*Tu respuesta:*

Si memoriza bastante. Train 0.991 y val 0.829 hueco 0.16, se aprende de memoria el train casi perfecto y en val baja.

**Tarea 5 · XGBoost con frenos.** Pon:
- `max_depth=3`: árboles pequeños.
- `learning_rate=0.05`: poco caso a cada árbol.
- `n_estimators=300`: como le hacemos poco caso a cada uno, hacen falta más árboles.

In [33]:
xgb_frenos = XGBClassifier(max_depth=3, learning_rate=0.05, n_estimators=300, random_state=SEMILLA)
resultados.append(evaluar("XGBoost con frenos", xgb_frenos))
mostrar(resultados)

,modelo,AUC val,± AUC val,AUC train
0,Baseline (no ordena),0.500,NaN,0.500
1,Árbol sin límite,0.735,0.028,0.998
2,Random Forest,0.823,0.047,0.996
3,AdaBoost,0.832,0.022,0.863
4,XGBoost por defecto,0.829,0.042,0.991
5,XGBoost con frenos,0.851,0.033,0.935


**Fíjate:** Compáralo con el XGBoost por defecto. ¿Memoriza menos? ¿Mejora `AUC val`?

*Tu respuesta:*

Si memoriza menos, hueco de 0.16 a 0.08 y mejora val de 0.829 a 0.851 aunque 0.02 es menos que el ± pero va mejor.

## 3 · ¿Cuál elegimos?

Esta celda ya está hecha: ejecútala. Ordena la tabla: el mejor queda arriba.

In [34]:
mostrar(pd.DataFrame(resultados).sort_values("AUC val", ascending=False))

,modelo,AUC val,± AUC val,AUC train
5,XGBoost con frenos,0.851,0.033,0.935
3,AdaBoost,0.832,0.022,0.863
4,XGBoost por defecto,0.829,0.042,0.991
2,Random Forest,0.823,0.047,0.996
1,Árbol sin límite,0.735,0.028,0.998
0,Baseline (no ordena),0.500,NaN,0.500


Contesta:
1. ¿Qué modelo gana?
2. Resta su `AUC val` y el del segundo. ¿La diferencia es mayor que el `±`? ¿Gana o es un empate?
3. ¿Cuál eliges? Si hay empate, elige el más sencillo o el más rápido.

*Tu respuesta:*

AdaBoost tiene 200 arbolitos de 1 pregunta, XGBoost con frenos tiene 300 árboles de profundidad 3, es más pesado. Como están empatados en 0.85 vs 0.83 con el ±, elijo AdaBoost porque es más sencillo y rápido y casi no memoriza, hueco 0.03

## 4 · El test, una sola vez

Esta celda ya está hecha: ejecútala. Entrena el Random Forest y el XGBoost con frenos con todo el train y los prueba en el test, que hasta ahora no habíamos tocado.

In [35]:
rf = RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(X_train, y_train)
xgb_frenos.fit(X_train, y_train)

# predict_proba(...)[:, 1] es la probabilidad de la clase 1
print("Random Forest      · AUC test:", round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 3))
print("XGBoost con frenos · AUC test:", round(roc_auc_score(y_test, xgb_frenos.predict_proba(X_test)[:, 1]), 3))

Random Forest      · AUC test: 0.863
XGBoost con frenos · AUC test: 0.9


**Fíjate:** ¿El resultado del test se parece al de validación (`AUC val`)? Si se parece, la validación nos dijo la verdad.

*Tu respuesta:*

Sí se parece. RF 0.823 en val y 0.863 en test, XGBoost 0.851 en val y 0.9 en test. Sube un poco en test pero está cerca y dentro del ±, así que la validación sí nos dijo la verdad.

## 5 · Conclusión

En dos o tres frases: ¿qué modelo eliges? ¿Por qué? Usa números de la tabla.

*Tu respuesta:*

Si hay que elegir por sencillez al ser empate 0.851 ±0.033 vs 0.832 ±0.022 dif 0.019 < ±, elijo AdaBoost con 0.832 en val y train 0.863 hueco 0.03 que casi no memoriza y es 10 veces más ligero, aunque si quisieramos priorizar por valor total XGBoost tiene metricas ligeramente mejores

## 6 ·

Tres pruebas cortas. Van en una tabla aparte, `extra`, para no mezclarlas con la de antes.

**Extra 1 · ¿Cuántos árboles hacen falta?** Esta celda ya está hecha: ejecútala. Entrena Random Forest con 10, 50 y 100 árboles.

In [36]:
extra = []
for n in [10, 50, 100]:
    extra.append(evaluar(f"Random Forest de {n} árboles", RandomForestClassifier(n_estimators=n, random_state=SEMILLA, n_jobs=-1)))
mostrar(extra)

,modelo,AUC val,± AUC val,AUC train
0,Random Forest de 10 árboles,0.817,0.032,0.992
1,Random Forest de 50 árboles,0.823,0.040,0.996
2,Random Forest de 100 árboles,0.823,0.041,0.996


**Fíjate:** ¿Mejora al pasar de 10 a 50 árboles? ¿Y de 50 a 100? ¿Empeora alguna vez al poner más árboles?

*Tu respuesta:*

De 10 a 50 mejora 0.006 que es menos que el ± así que es empate, de 50 a 100 ya no mejora se queda en 0.823. No empeora al poner más árboles, se estabiliza

**Extra 2 · Un árbol con límite.** Un solo árbol, pero de 5 preguntas como máximo (`max_depth=5`). Cambia los `___`.

In [37]:
extra.append(evaluar("Árbol de 5 preguntas", DecisionTreeClassifier(max_depth=5, random_state=SEMILLA)))
mostrar(extra)

,modelo,AUC val,± AUC val,AUC train
0,Random Forest de 10 árboles,0.817,0.032,0.992
1,Random Forest de 50 árboles,0.823,0.040,0.996
2,Random Forest de 100 árboles,0.823,0.041,0.996
3,Árbol de 5 preguntas,0.817,0.021,0.886


**Fíjate:** Compáralo con el árbol sin límite de la tabla de antes. ¿Memoriza menos? ¿Llega a los ensembles?

*Tu respuesta:*

Sí memoriza mucho menos. Sin límite hueco 0.26 (0.996-0.735) con 5 preguntas hueco 0.06 (0.886-0.817). Val sube de 0.735 a 0.817. Llega a los ensemble, 0.817 vs 0.823 del RF dif 0.006 < ± es empate

**Extra 3 · Mucho caso a cada árbol.** El mismo XGBoost con frenos, pero con `learning_rate=0.5` en lugar de 0,05. Cambia el `___`.

In [38]:
xgb_rapido = XGBClassifier(max_depth=3, learning_rate=0.5, n_estimators=300, random_state=SEMILLA)
extra.append(evaluar("XGBoost con learning_rate 0,5", xgb_rapido))
mostrar(extra)

,modelo,AUC val,± AUC val,AUC train
0,Random Forest de 10 árboles,0.817,0.032,0.992
1,Random Forest de 50 árboles,0.823,0.040,0.996
2,Random Forest de 100 árboles,0.823,0.041,0.996
3,Árbol de 5 preguntas,0.817,0.021,0.886
4,"XGBoost con learning_rate 0,5",0.831,0.036,0.991


**Fíjate:** Compáralo con el XGBoost con frenos (0,05). ¿Memoriza más? ¿Mejora o empeora `AUC val`?

*Tu respuesta:*

Memoriza más. Con 0,05 train 0,935 hueco 0,08 con 0,5 train 0,991 hueco 0,16. AUC val empeora de 0,851 a 0,831 baja 0,02 pero es casi empate porque 0,02 < ± 0,033 y 0,036